# Stage 7 — Fact-Checking with Credible-Source Enforcement
**Tadhkirat Dawood Al-Antaqi Project**

Validates herb ↔ disease treatment claims from `disease_treatment_english_v1.xlsx` against modern sources, using `gemini-3.5-flash` + Google Search grounding, with a real credibility check on the URLs actually returned (not just prompt trust).

No GPU needed — this notebook only makes API calls to Vertex AI. A standard CPU runtime is sufficient.

## AUTH & CLIENT SETUP

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project='your-gcp-project-id',
    location='global',   # required for gemini-3.5-flash, same as gemini-3-flash-preview
    http_options=types.HttpOptions(api_version="v1"),
)

MODEL = "gemini-3.5-flash"

# -----------------------------------------------------------------------------

## CREDIBLE DOMAIN ALLOW-LIST (post-hoc verification, not a search filter)

In [ ]:
# Vertex's Google Search grounding tool only supports exclude_domains, not an
# allow-list, so credibility is enforced by checking the ACTUAL citation URLs
# returned in groundingMetadata against this list.
# -----------------------------------------------------------------------------
CREDIBLE_DOMAIN_PATTERNS = [
    # Medical / scientific literature databases
    "ncbi.nlm.nih.gov", "pubmed.ncbi.nlm.nih.gov", "nih.gov", "who.int",
    "cochranelibrary.com", "sciencedirect.com", "springer.com", "springerlink.com",
    "wiley.com", "onlinelibrary.wiley.com", "mdpi.com", "frontiersin.org",
    "nature.com", "cell.com", "plos.org", "tandfonline.com", "jstor.org",
    "bmj.com", "thelancet.com", "jamanetwork.com", "elsevier.com",
    "researchgate.net", "sciencedaily.com", "academic.oup.com",
    # Botanical / taxonomic authorities
    "gbif.org", "kew.org", "worldfloraonline.org", "powo.science.kew.org",
    "ipni.org", "tropicos.org", "efloras.org", "plantsoftheworldonline.org",
    # Government / institutional (broad patterns)
    ".gov", ".edu",
    "ema.europa.eu", "efsa.europa.eu", "fda.gov",
]

# Domains known to be low-quality / unreliable for medical claims —
# used to seed exclude_domains on the FIRST pass already.
KNOWN_LOW_QUALITY_DOMAINS = [
    "pinterest.com", "quora.com", "reddit.com", "answers.com",
    "healthline.com",  # decent for general audience but not primary source; excluded to force primary lit
    "webmd.com",        # same reasoning
    "wikihow.com",
]

def is_credible_url(url: str) -> bool:
    url = (url or "").lower()
    return any(pattern in url for pattern in CREDIBLE_DOMAIN_PATTERNS)

# -----------------------------------------------------------------------------

## PROMPTS (hybrid Arabic/English, consistent with project convention)

In [ ]:
def build_prompt(arabic_name, sci_name, disease_en, treatment_en, effect_en, strict=False):
    strict_clause = ""
    if strict:
        strict_clause = """
تنبيه: المحاولة الأولى فشلت في العثور على مصدر علمي موثوق. في هذه المحاولة،
ابحث حصرياً في المصادر التالية: الأبحاث المنشورة في PubMed أو NCBI، منظمة الصحة
العالمية (WHO)، المجلات العلمية المحكمة (Elsevier, Springer, Wiley, MDPI, Frontiers,
Nature)، أو المواقع الحكومية (.gov) والأكاديمية (.edu). تجاهل المواقع الصحية العامة
أو المدونات أو المنتديات تماماً.
"""

    return f"""أنت صيدلاني وباحث علمي متخصص في التحقق من صحة الادعاءات الطبية التقليدية
مقابل الأدلة العلمية الحديثة.

النبات: {arabic_name} ({sci_name})
الادعاء من كتاب تذكرة داود الأنطاكي:
- الحالة/المرض: {disease_en}
- طريقة العلاج: {treatment_en}
- الأثر المتوقع: {effect_en}
{strict_clause}
ابحث على الإنترنت عن أدلة علمية حديثة (أبحاث محكمة، قواعد بيانات طبية، منظمات صحية
رسمية) تؤيد أو تدحض فعالية هذا النبات لهذه الحالة تحديداً.

قواعد التقييم:
١. اعتمد فقط على مصادر أولية موثوقة: أبحاث محكمة، PubMed/NCBI، WHO، مواقع .gov أو .edu،
   مجلات علمية معروفة. لا تعتمد على مواقع صحية عامة أو مدونات أو منتديات.
٢. إذا لم تجد أي مصدر موثوق يتناول هذه الحالة تحديداً، صرّح بذلك بوضوح ولا تخترع دليلاً.
٣. اختلاف طريقة التحضير أو الجرعة عن الاستخدام الحديث لا يعني عدم الفعالية، طالما
   النبات نفسه مثبت الفعالية لهذه الحالة.

Respond ONLY in this exact format with no extra text:
VERDICT: [Strongly Agree/Agree/Neutral/Disagree/Strongly Disagree/Insufficient Evidence]
CONFIDENCE: [0.0 to 1.0]
REASON: [one to two sentences in Arabic explaining the verdict and citing which source type supported it]
"""

# -----------------------------------------------------------------------------

## SINGLE ROW FACT-CHECK (with grounding + credibility check + 1 retry)

In [ ]:
import re

def parse_response(text):
    verdict = re.search(r"VERDICT:\s*(.+)", text)
    confidence = re.search(r"CONFIDENCE:\s*([\d.]+)", text)
    reason = re.search(r"REASON:\s*(.+)", text, re.DOTALL)
    return {
        "verdict": verdict.group(1).strip() if verdict else "PARSE_ERROR",
        "confidence": float(confidence.group(1)) if confidence else 0.0,
        "reason": reason.group(1).strip() if reason else "",
    }

def extract_grounding_urls(response):
    urls = []
    try:
        chunks = response.candidates[0].grounding_metadata.grounding_chunks
        for c in chunks:
            if hasattr(c, "web") and c.web and c.web.uri:
                urls.append(c.web.uri)
    except (AttributeError, IndexError, TypeError):
        pass
    return urls

def factcheck_row(arabic_name, sci_name, disease_en, treatment_en, effect_en,
                   exclude_domains=None):
    exclude_domains = list(exclude_domains or KNOWN_LOW_QUALITY_DOMAINS)

    # --- PASS 1 ---
    tool = types.Tool(google_search=types.GoogleSearch(exclude_domains=exclude_domains))
    config = types.GenerateContentConfig(tools=[tool])
    prompt = build_prompt(arabic_name, sci_name, disease_en, treatment_en, effect_en, strict=False)

    resp = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    urls = extract_grounding_urls(resp)
    credible_urls = [u for u in urls if is_credible_url(u)]
    parsed = parse_response(resp.text)

    if credible_urls:
        parsed.update({
            "credible_sources": credible_urls,
            "all_sources": urls,
            "credibility_check": "Passed (pass 1)",
            "retried": False,
        })
        return parsed

    # --- PASS 2 (retry): exclude whatever non-credible domains showed up, tighten prompt ---
    retry_exclude = list(set(exclude_domains) | {u.split("/")[2] for u in urls if "/" in u})
    tool2 = types.Tool(google_search=types.GoogleSearch(exclude_domains=retry_exclude))
    config2 = types.GenerateContentConfig(tools=[tool2])
    prompt2 = build_prompt(arabic_name, sci_name, disease_en, treatment_en, effect_en, strict=True)

    resp2 = client.models.generate_content(model=MODEL, contents=prompt2, config=config2)
    urls2 = extract_grounding_urls(resp2)
    credible_urls2 = [u for u in urls2 if is_credible_url(u)]
    parsed2 = parse_response(resp2.text)

    if credible_urls2:
        parsed2.update({
            "credible_sources": credible_urls2,
            "all_sources": urls2,
            "credibility_check": "Passed (pass 2 / retry)",
            "retried": True,
        })
        return parsed2

    # --- Still nothing credible after retry: force verdict, flag for review ---
    return {
        "verdict": "Insufficient Evidence",
        "confidence": 0.0,
        "reason": "لم يتم العثور على مصدر علمي موثوق بعد محاولتين بحث.",
        "credible_sources": [],
        "all_sources": urls + urls2,
        "credibility_check": "Failed after retry",
        "retried": True,
    }

# -----------------------------------------------------------------------------

## MAIN LOOP: resume-safe, saves after every row (project convention)

In [ ]:
import pandas as pd
import os
import time

DRIVE_DIR = "/content/drive/MyDrive/tadhkirat_dawood/"
INPUT_XLSX = DRIVE_DIR + "disease_treatment_english_v1.xlsx"
IDENTIFICATION_CSV = DRIVE_DIR + "plant_identification_v2.csv"
OUTPUT_CSV = DRIVE_DIR + "factcheck_credible_v1.csv"
PROGRESS_FILE = DRIVE_DIR + "factcheck_progress_v1.txt"

df = pd.read_excel(INPUT_XLSX)          # 7,362 rows: entry_id, arabic_name, disease_or_condition_en, ...
ident = pd.read_csv(IDENTIFICATION_CSV) # for scientific_name, linked by entry_id

sci_name_map = dict(zip(ident["entry_id"], ident.get("scientific_name", ident["entry_id"])))

# Resume: load existing output if present
if os.path.exists(OUTPUT_CSV):
    done_df = pd.read_csv(OUTPUT_CSV)
    done_ids = set(zip(done_df["entry_id"], done_df["disease_or_condition_en"]))
    results = done_df.to_dict("records")
else:
    done_ids = set()
    results = []

for i, row in df.iterrows():
    key = (row["entry_id"], row["disease_or_condition_en"])
    if key in done_ids:
        continue

    sci_name = sci_name_map.get(row["entry_id"], "")

    try:
        result = factcheck_row(
            arabic_name=row["arabic_name"],
            sci_name=sci_name,
            disease_en=row["disease_or_condition_en"],
            treatment_en=row["treatment_method_en"],
            effect_en=row["expected_effect_en"],
        )
    except Exception as e:
        result = {
            "verdict": "ERROR", "confidence": 0.0, "reason": str(e),
            "credible_sources": [], "all_sources": [],
            "credibility_check": "Error", "retried": False,
        }

    record = {
        "entry_id": row["entry_id"],
        "arabic_name": row["arabic_name"],
        "scientific_name": sci_name,
        "disease_or_condition_en": row["disease_or_condition_en"],
        "treatment_method_en": row["treatment_method_en"],
        "verdict": result["verdict"],
        "confidence": result["confidence"],
        "reason": result["reason"],
        "credible_sources": " | ".join(result["credible_sources"]),
        "credibility_check": result["credibility_check"],
        "retried": result["retried"],
        "needs_human_review": result["credibility_check"] == "Failed after retry",
    }
    results.append(record)

    # save after every single row (project convention — no data loss on disconnect)
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    with open(PROGRESS_FILE, "w") as f:
        f.write(f"{i+1}/{len(df)} rows processed\n")

    time.sleep(0.3)  # light throttle, adjust if hitting rate limits

print(f"Done. {len(results)} rows written to {OUTPUT_CSV}")